# Lábia — teste de fumaça do ambiente do Kaggle

Um **teste de fumaça** é a primeira verificação, a mais barata e a mais burra: você liga o aparelho e vê se sai fumaça. O nome vem da eletrônica. Ele não prova que tudo funciona; prova que nada está obviamente quebrado.

Rodamos este notebook **antes** de gastar qualquer hora de GPU com treino, por três motivos:

1. confirmar que a conta tem mesmo acesso a GPU (a verificação por telefone funcionou);
2. anotar exatamente **qual hardware** nos foi dado, porque toda métrica de latência do projeto só significa alguma coisa junto da descrição da máquina;
3. descobrir agora, e não na véspera da entrega, se alguma peça essencial está faltando.

Custo esperado: menos de dois minutos de GPU.

## 1. A GPU existe e está visível?

`nvidia-smi` é o programa que a própria NVIDIA instala junto do driver. Ele lista as placas que o sistema enxerga. Se este comando falhar, não há GPU nenhuma ligada nesta sessão, e o resto do notebook não faz sentido.

In [ ]:
!nvidia-smi

## 2. O PyTorch enxerga a GPU?

Ver a placa no sistema não basta: a biblioteca que vai treinar o modelo precisa conseguir usá-la.

- **PyTorch** é a biblioteca de redes neurais que usamos para treinar.
- **CUDA** é a camada da NVIDIA que deixa um programa rodar contas na GPU em vez de no processador. Se o PyTorch foi compilado para uma versão de CUDA diferente da do driver, ele simplesmente não vê a placa.
- **VRAM** é a memória da própria GPU. É o limite real do nosso projeto: o modelo precisa caber nela.

O plano prevê **2 placas T4, com 16 GB de VRAM cada**. O número que sai aqui é o que vamos citar no README.

In [ ]:
import torch

print(f"PyTorch:            {torch.__version__}")
print(f"CUDA disponível:    {torch.cuda.is_available()}")
print(f"CUDA do PyTorch:    {torch.version.cuda}")
print(f"Número de GPUs:     {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    propriedades = torch.cuda.get_device_properties(i)
    vram_gb = propriedades.total_memory / 1024**3
    print(
        f"  GPU {i}: {propriedades.name} | VRAM {vram_gb:.1f} GB | capacidade {propriedades.major}.{propriedades.minor}"
    )

## 3. A GPU realmente calcula?

Aparecer na lista não prova que funciona. Aqui multiplicamos duas matrizes grandes na GPU e conferimos o resultado contra o processador.

Multiplicação de matrizes é a conta que domina o treino de uma rede neural: quase todo o tempo de treino é isso, repetido milhões de vezes.

A comparação usa uma tolerância, e não igualdade exata, porque a GPU faz contas com menos casas decimais em troca de velocidade. Diferenças minúsculas são normais e esperadas.

In [ ]:
import time

import torch

assert torch.cuda.is_available(), "Sem GPU nesta sessão: confira o acelerador nas opções do notebook."

torch.manual_seed(42)  # semente fixa: o projeto inteiro precisa ser reproduzível
a = torch.randn(4096, 4096)
b = torch.randn(4096, 4096)

inicio = time.perf_counter()
esperado = a @ b
segundos_cpu = time.perf_counter() - inicio

a_gpu, b_gpu = a.cuda(), b.cuda()
torch.cuda.synchronize()  # a GPU trabalha em segundo plano; sem isso mediríamos tempo errado
inicio = time.perf_counter()
obtido = a_gpu @ b_gpu
torch.cuda.synchronize()
segundos_gpu = time.perf_counter() - inicio

print(f"CPU: {segundos_cpu:.3f} s")
print(f"GPU: {segundos_gpu:.3f} s  ({segundos_cpu / segundos_gpu:.0f}x mais rápido)")

assert torch.allclose(esperado, obtido.cpu(), atol=1e-3), "A GPU devolveu um resultado errado."
print("\nResultado da GPU confere com o da CPU.")

## 4. As duas GPUs funcionam?

O Kaggle entrega **duas** T4. Elas não somam a memória: são 16 GB + 16 GB, e não 32 GB. Mas dá para colocar metade das camadas do modelo em cada uma, que é como pretendemos rodar o professor.

Aqui só confirmamos que a segunda placa também calcula.

In [ ]:
import torch

for i in range(torch.cuda.device_count()):
    x = torch.randn(1024, 1024, device=f"cuda:{i}")
    y = (x @ x).sum().item()
    livre, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: calculou (soma = {y:.1f}) | livre {livre / 1024**3:.1f} GB de {total / 1024**3:.1f} GB")

## 5. Quanto temos de processador, memória e disco?

Esses números entram no projeto por motivos concretos:

- **RAM** é a memória do computador, separada da VRAM. O modelo é carregado aqui antes de ir para a placa, então uma RAM pequena pode travar o carregamento mesmo com VRAM sobrando.
- **Disco** é o limite para baixar os arquivos do modelo. O professor que pretendemos usar ocupa dezenas de gigabytes.
- **Núcleos de CPU** importam porque parte do trabalho (preparar os dados) não usa GPU.

In [ ]:
import os
import platform
import shutil

print(f"Python:          {platform.python_version()}")
print(f"Sistema:         {platform.platform()}")
print(f"Núcleos de CPU:  {os.cpu_count()}")

with open("/proc/meminfo") as arquivo:
    for linha in arquivo:
        if linha.startswith(("MemTotal", "MemAvailable")):
            rotulo, valor_kb = linha.split(":")
            print(f"{rotulo + ':':16} {int(valor_kb.strip().split()[0]) / 1024**2:.1f} GB")

for caminho, nome in [("/kaggle/working", "disco de trabalho"), ("/kaggle/temp", "disco temporário")]:
    if os.path.exists(caminho):
        uso = shutil.disk_usage(caminho)
        print(f"{nome:18} {uso.free / 1024**3:.0f} GB livres de {uso.total / 1024**3:.0f} GB")

## 6. As bibliotecas essenciais já vêm instaladas?

O Kaggle já traz muita coisa pronta. Saber o que falta importa porque este notebook roda **sem internet**: tudo o que não vier instalado precisa ser previsto antes.

- **transformers**: carrega e roda modelos de linguagem.
- **peft** e **trl**: fazem o ajuste fino do modelo (treinar só uma parte pequena dele).
- **bitsandbytes**: comprime o modelo para caber na memória da placa.
- **accelerate**: distribui o trabalho entre as duas GPUs.

Nenhuma falha aqui é fatal: o que faltar a gente instala no notebook de treino, com a internet ligada.

In [ ]:
import importlib.metadata as metadados

for pacote in ["torch", "transformers", "accelerate", "peft", "trl", "datasets", "bitsandbytes", "vllm"]:
    try:
        print(f"{pacote:16} {metadados.version(pacote)}")
    except metadados.PackageNotFoundError:
        print(f"{pacote:16} AUSENTE (instalar no notebook de treino)")

## 7. Resumo da máquina

Guardamos o resultado num arquivo. A regra do projeto é que **nenhum número vai para o README sem sair de um script que roda**, e a descrição do hardware vale a mesma regra: toda medida de latência só tem sentido ao lado da máquina que a produziu.

In [ ]:
import json
import os
import platform
from datetime import UTC, datetime

import torch

ambiente = {
    "kind": "measured",
    "generated_at": datetime.now(UTC).isoformat(),
    "host": "kaggle",
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cpu_cores": os.cpu_count(),
    "gpus": [
        {
            "name": torch.cuda.get_device_properties(i).name,
            "vram_gb": round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 1),
        }
        for i in range(torch.cuda.device_count())
    ],
}

with open("/kaggle/working/ambiente.json", "w", encoding="utf-8") as arquivo:
    json.dump(ambiente, arquivo, ensure_ascii=False, indent=2)

print(json.dumps(ambiente, ensure_ascii=False, indent=2))
print("\nTeste de fumaça concluído.")